In [5]:
asm_code = """
.org 0x0000
.text

start:  add x1, x2, x3
        sub x4, x1, x5
        addi x6, x0, 10
        lw x7, 0(x1)
        sw x7, 4(x1)
        beq x1, x2, end
        bne x1, x3, start

.data
val1:   .word 25
val2:   .byte 7

.text
end:    or x8, x1, x2
        .end
"""

with open("input.asm", "w", encoding="utf-8") as f:
    f.write(asm_code)

print("input.asm oluşturuldu")

input.asm oluşturuldu


In [7]:
import os

print("Bulunduğun klasör:", os.getcwd())
print("Klasördeki dosyalar:", os.listdir())

Bulunduğun klasör: C:\Users\monster\picorv_assembler
Klasördeki dosyalar: ['.ipynb_checkpoints', 'input.asm', 'Untitled.ipynb']


In [9]:
import sys


class OpcodeNode:
    def __init__(self, mnemonic, fmt, opcode, funct3="", funct7=""):
        self.mnemonic = mnemonic
        self.fmt = fmt
        self.opcode = opcode
        self.funct3 = funct3
        self.funct7 = funct7
        self.next = None


class OpcodeTable:
    def __init__(self):
        self.head = None

    def add(self, mnemonic, fmt, opcode, funct3="", funct7=""):
        new_node = OpcodeNode(mnemonic, fmt, opcode, funct3, funct7)
        if self.head is None:
            self.head = new_node
            return

        current = self.head
        while current.next is not None:
            current = current.next
        current.next = new_node

    def find(self, mnemonic):
        current = self.head
        while current is not None:
            if current.mnemonic == mnemonic:
                return current
            current = current.next
        return None


class SymbolNode:
    def __init__(self, label, address):
        self.label = label
        self.address = address
        self.next = None


class SymbolTable:
    def __init__(self):
        self.head = None

    def add(self, label, address):
        if self.find(label) is not None:
            raise ValueError(f"Yinelenen label: {label}")

        new_node = SymbolNode(label, address)
        if self.head is None:
            self.head = new_node
            return

        current = self.head
        while current.next is not None:
            current = current.next
        current.next = new_node

    def find(self, label):
        current = self.head
        while current is not None:
            if current.label == label:
                return current
            current = current.next
        return None

    def display(self):
        current = self.head
        print("\nSymbol Table")
        print("-" * 30)
        while current is not None:
            print(f"{current.label:15} -> 0x{current.address:08X}")
            current = current.next
        print("-" * 30)


class Assembler:
    def __init__(self):
        self.opcodes = OpcodeTable()
        self.symbols = SymbolTable()
        self.start_address = 0
        self.current_segment = ".text"
        self._load_opcodes()

    def _load_opcodes(self):
        # R-Type
        self.opcodes.add("add", "R", "0110011", "000", "0000000")
        self.opcodes.add("sub", "R", "0110011", "000", "0100000")
        self.opcodes.add("and", "R", "0110011", "111", "0000000")
        self.opcodes.add("or",  "R", "0110011", "110", "0000000")
        self.opcodes.add("xor", "R", "0110011", "100", "0000000")
        self.opcodes.add("sll", "R", "0110011", "001", "0000000")
        self.opcodes.add("srl", "R", "0110011", "101", "0000000")

        # I-Type
        self.opcodes.add("addi", "I", "0010011", "000")
        self.opcodes.add("lw",   "I", "0000011", "010")
        self.opcodes.add("jalr", "I", "1100111", "000")

        # S-Type
        self.opcodes.add("sw",   "S", "0100011", "010")

        # B-Type
        self.opcodes.add("beq",  "B", "1100011", "000")
        self.opcodes.add("bne",  "B", "1100011", "001")

    def clean_line(self, line):
        comment_pos_hash = line.find("#")
        comment_pos_semicolon = line.find(";")

        cut_pos = -1
        if comment_pos_hash != -1 and comment_pos_semicolon != -1:
            cut_pos = min(comment_pos_hash, comment_pos_semicolon)
        elif comment_pos_hash != -1:
            cut_pos = comment_pos_hash
        elif comment_pos_semicolon != -1:
            cut_pos = comment_pos_semicolon

        if cut_pos != -1:
            line = line[:cut_pos]

        return line.strip()

    def split_label(self, line):
        if ":" in line:
            parts = line.split(":", 1)
            label = parts[0].strip()
            rest = parts[1].strip()
            return label, rest
        return None, line

    def parse_register(self, token):
        token = token.strip()
        if not token.startswith("x"):
            raise ValueError(f"Geçersiz register: {token}")

        num_str = token[1:]
        if not num_str.isdigit():
            raise ValueError(f"Geçersiz register: {token}")

        num = int(num_str)
        if num < 0 or num > 31:
            raise ValueError(f"Register aralık dışı: {token}")

        return format(num, "05b")

    def parse_immediate(self, token):
        token = token.strip()
        return int(token, 0)

    def to_twos_complement(self, value, bits):
        mask = (1 << bits) - 1
        return value & mask

    def split_operands(self, operand_text):
        # Burada geçici Python split kullanılıyor ama ana veri yapıları linked list mantığıyla kuruldu.
        # İstenirse bunu da tamamen manuel parser ile ikinci sürümde yapabiliriz.
        parts = operand_text.split(",")
        cleaned = []
        for part in parts:
            cleaned.append(part.strip())
        return cleaned

    def parse_memory_operand(self, text):
        text = text.strip()
        left_paren = text.find("(")
        right_paren = text.find(")")

        if left_paren == -1 or right_paren == -1 or right_paren < left_paren:
            raise ValueError(f"Geçersiz bellek operandı: {text}")

        imm_text = text[:left_paren].strip()
        reg_text = text[left_paren + 1:right_paren].strip()

        if imm_text == "":
            imm_value = 0
        else:
            imm_value = self.parse_immediate(imm_text)

        rs1 = self.parse_register(reg_text)
        return imm_value, rs1

    def is_directive(self, text):
        return text.startswith(".")

    def instruction_size(self, line):
        text = line.strip()
        if text == "":
            return 0

        if text.startswith(".word"):
            return 4
        if text.startswith(".byte"):
            return 1
        if text.startswith(".data") or text.startswith(".text") or text.startswith(".end"):
            return 0
        if text.startswith(".org"):
            return 0

        return 4

    def pass1(self, input_file):
        pc = self.start_address
        self.current_segment = ".text"

        with open(input_file, "r", encoding="utf-8") as f:
            line_no = 0
            for raw_line in f:
                line_no += 1
                line = self.clean_line(raw_line)
                if line == "":
                    continue

                label, rest = self.split_label(line)

                if label is not None:
                    self.symbols.add(label, pc)

                if rest == "":
                    continue

                if self.is_directive(rest):
                    if rest.startswith(".org"):
                        parts = rest.split(maxsplit=1)
                        if len(parts) != 2:
                            raise ValueError(f"Satır {line_no}: .org parametresi eksik")
                        pc = self.parse_immediate(parts[1])
                        continue

                    if rest.startswith(".data"):
                        self.current_segment = ".data"
                        continue

                    if rest.startswith(".text"):
                        self.current_segment = ".text"
                        continue

                    if rest.startswith(".end"):
                        break

                    pc += self.instruction_size(rest)
                    continue

                pc += 4

    def encode_r_type(self, op_info, operands):
        if len(operands) != 3:
            raise ValueError("R-Type komut için 3 operand gerekli")

        rd = self.parse_register(operands[0])
        rs1 = self.parse_register(operands[1])
        rs2 = self.parse_register(operands[2])

        return (
            op_info.funct7 +
            rs2 +
            rs1 +
            op_info.funct3 +
            rd +
            op_info.opcode
        )

    def encode_i_type(self, mnemonic, op_info, operands):
        if mnemonic == "lw":
            if len(operands) != 2:
                raise ValueError("lw için 2 operand gerekli")

            rd = self.parse_register(operands[0])
            imm_value, rs1 = self.parse_memory_operand(operands[1])
            imm = format(self.to_twos_complement(imm_value, 12), "012b")

            return imm + rs1 + op_info.funct3 + rd + op_info.opcode

        if mnemonic == "addi":
            if len(operands) != 3:
                raise ValueError("addi için 3 operand gerekli")

            rd = self.parse_register(operands[0])
            rs1 = self.parse_register(operands[1])
            imm_value = self.parse_immediate(operands[2])
            imm = format(self.to_twos_complement(imm_value, 12), "012b")

            return imm + rs1 + op_info.funct3 + rd + op_info.opcode

        if mnemonic == "jalr":
            if len(operands) != 3:
                raise ValueError("jalr için 3 operand gerekli")

            rd = self.parse_register(operands[0])
            rs1 = self.parse_register(operands[1])
            imm_value = self.parse_immediate(operands[2])
            imm = format(self.to_twos_complement(imm_value, 12), "012b")

            return imm + rs1 + op_info.funct3 + rd + op_info.opcode

        raise ValueError(f"Desteklenmeyen I-Type komut: {mnemonic}")

    def encode_s_type(self, op_info, operands):
        if len(operands) != 2:
            raise ValueError("S-Type komut için 2 operand gerekli")

        rs2 = self.parse_register(operands[0])
        imm_value, rs1 = self.parse_memory_operand(operands[1])

        imm12 = self.to_twos_complement(imm_value, 12)
        imm_bin = format(imm12, "012b")

        imm_high = imm_bin[:7]
        imm_low = imm_bin[7:]

        return (
            imm_high +
            rs2 +
            rs1 +
            op_info.funct3 +
            imm_low +
            op_info.opcode
        )

    def encode_b_type(self, op_info, operands, pc):
        if len(operands) != 3:
            raise ValueError("B-Type komut için 3 operand gerekli")

        rs1 = self.parse_register(operands[0])
        rs2 = self.parse_register(operands[1])

        label_or_imm = operands[2].strip()
        symbol = self.symbols.find(label_or_imm)

        if symbol is not None:
            target_address = symbol.address
            offset = target_address - pc
        else:
            offset = self.parse_immediate(label_or_imm)

        imm13 = self.to_twos_complement(offset, 13)

        bit12 = (imm13 >> 12) & 0b1
        bit11 = (imm13 >> 11) & 0b1
        bits10_5 = (imm13 >> 5) & 0b111111
        bits4_1 = (imm13 >> 1) & 0b1111

        return (
            format(bit12, "01b") +
            format(bits10_5, "06b") +
            rs2 +
            rs1 +
            op_info.funct3 +
            format(bits4_1, "04b") +
            format(bit11, "01b") +
            op_info.opcode
        )

    def encode_instruction(self, text, pc):
        if " " in text:
            mnemonic, operand_text = text.split(None, 1)
        else:
            mnemonic = text.strip()
            operand_text = ""

        op_info = self.opcodes.find(mnemonic)
        if op_info is None:
            raise ValueError(f"Geçersiz komut: {mnemonic}")

        operands = self.split_operands(operand_text) if operand_text != "" else []

        if op_info.fmt == "R":
            return self.encode_r_type(op_info, operands)

        if op_info.fmt == "I":
            return self.encode_i_type(mnemonic, op_info, operands)

        if op_info.fmt == "S":
            return self.encode_s_type(op_info, operands)

        if op_info.fmt == "B":
            return self.encode_b_type(op_info, operands, pc)

        raise ValueError(f"Bilinmeyen komut formatı: {op_info.fmt}")

    def process_directive_pass2(self, text, output_handle, pc):
        if text.startswith(".data") or text.startswith(".text"):
            return pc, False

        if text.startswith(".org"):
            parts = text.split(maxsplit=1)
            if len(parts) != 2:
                raise ValueError(".org parametresi eksik")
            pc = self.parse_immediate(parts[1])
            return pc, False

        if text.startswith(".word"):
            parts = text.split(maxsplit=1)
            if len(parts) != 2:
                raise ValueError(".word parametresi eksik")

            value = self.parse_immediate(parts[1])
            binary = format(self.to_twos_complement(value, 32), "032b")
            output_handle.write(binary + "\n")
            return pc + 4, False

        if text.startswith(".byte"):
            parts = text.split(maxsplit=1)
            if len(parts) != 2:
                raise ValueError(".byte parametresi eksik")

            value = self.parse_immediate(parts[1])
            binary = format(self.to_twos_complement(value, 8), "08b")
            output_handle.write(binary + "\n")
            return pc + 1, False

        if text.startswith(".end"):
            return pc, True

        raise ValueError(f"Desteklenmeyen direktif: {text}")

    def pass2(self, input_file, output_file):
        pc = self.start_address

        with open(input_file, "r", encoding="utf-8") as fin, open(output_file, "w", encoding="utf-8") as fout:
            line_no = 0
            for raw_line in fin:
                line_no += 1
                line = self.clean_line(raw_line)

                if line == "":
                    continue

                label, rest = self.split_label(line)
                text = rest.strip() if label is not None else line

                if text == "":
                    continue

                try:
                    if self.is_directive(text):
                        pc, should_stop = self.process_directive_pass2(text, fout, pc)
                        if should_stop:
                            break
                        continue

                    machine_code = self.encode_instruction(text, pc)
                    fout.write(machine_code + "\n")
                    pc += 4

                except Exception as e:
                    raise ValueError(f"Satır {line_no}: {e}") from e

    def assemble(self, input_file, output_file):
        self.pass1(input_file)
        self.pass2(input_file, output_file)

        print("\nAssembly işlemi tamamlandı.")
        print(f"Girdi dosyası : {input_file}")
        print(f"Çıktı dosyası : {output_file}")
        self.symbols.display()


def main():
    if len(sys.argv) < 2:
        print("Kullanım: python assembler.py input.asm [output.txt]")
        return

    input_file = sys.argv[1]

    if len(sys.argv) >= 3:
        output_file = sys.argv[2]
    else:
        output_file = "output.txt"

    assembler = Assembler()

    try:
        assembler.assemble(input_file, output_file)
    except Exception as e:
        print(f"Hata: {e}")


if __name__ == "__main__":
    main()

Hata: [Errno 2] No such file or directory: '-f'


In [11]:
assembler = Assembler()
assembler.assemble("input.asm", "output.txt")


Assembly işlemi tamamlandı.
Girdi dosyası : input.asm
Çıktı dosyası : output.txt

Symbol Table
------------------------------
start           -> 0x00000000
val1            -> 0x0000001C
val2            -> 0x00000020
end             -> 0x00000021
------------------------------


In [13]:
import os


class OpcodeNode:
    def __init__(self, mnemonic, fmt, opcode, funct3="", funct7=""):
        self.mnemonic = mnemonic
        self.fmt = fmt
        self.opcode = opcode
        self.funct3 = funct3
        self.funct7 = funct7
        self.next = None


class OpcodeTable:
    def __init__(self):
        self.head = None

    def add(self, mnemonic, fmt, opcode, funct3="", funct7=""):
        new_node = OpcodeNode(mnemonic, fmt, opcode, funct3, funct7)

        if self.head is None:
            self.head = new_node
            return

        current = self.head
        while current.next is not None:
            current = current.next
        current.next = new_node

    def find(self, mnemonic):
        current = self.head
        while current is not None:
            if current.mnemonic == mnemonic:
                return current
            current = current.next
        return None

    def display(self):
        print("\nOpcode Table")
        print("-" * 70)
        print(f"{'Mnemonic':10} {'Format':8} {'Opcode':10} {'funct3':8} {'funct7':10}")
        print("-" * 70)

        current = self.head
        while current is not None:
            print(
                f"{current.mnemonic:10} "
                f"{current.fmt:8} "
                f"{current.opcode:10} "
                f"{current.funct3:8} "
                f"{current.funct7:10}"
            )
            current = current.next

        print("-" * 70)


class SymbolNode:
    def __init__(self, label, address):
        self.label = label
        self.address = address
        self.next = None


class SymbolTable:
    def __init__(self):
        self.head = None

    def add(self, label, address):
        if self.find(label) is not None:
            raise ValueError(f"Yinelenen label: {label}")

        new_node = SymbolNode(label, address)

        if self.head is None:
            self.head = new_node
            return

        current = self.head
        while current.next is not None:
            current = current.next
        current.next = new_node

    def find(self, label):
        current = self.head
        while current is not None:
            if current.label == label:
                return current
            current = current.next
        return None

    def display(self):
        print("\nSymbol Table")
        print("-" * 40)
        print(f"{'Label':20} {'Address':15}")
        print("-" * 40)

        current = self.head
        while current is not None:
            print(f"{current.label:20} 0x{current.address:08X}")
            current = current.next

        print("-" * 40)


class Assembler:
    def __init__(self):
        self.opcodes = OpcodeTable()
        self.symbols = SymbolTable()
        self.start_address = 0
        self.current_segment = ".text"
        self.supported_directives = ".data .text .word .byte .org .end"
        self._load_opcodes()

    # -------------------------------------------------
    # OPCODE TABLE
    # -------------------------------------------------
    def _load_opcodes(self):
        # R-Type
        self.opcodes.add("add", "R", "0110011", "000", "0000000")
        self.opcodes.add("sub", "R", "0110011", "000", "0100000")
        self.opcodes.add("and", "R", "0110011", "111", "0000000")
        self.opcodes.add("or",  "R", "0110011", "110", "0000000")
        self.opcodes.add("xor", "R", "0110011", "100", "0000000")
        self.opcodes.add("sll", "R", "0110011", "001", "0000000")
        self.opcodes.add("srl", "R", "0110011", "101", "0000000")

        # I-Type
        self.opcodes.add("addi", "I", "0010011", "000")
        self.opcodes.add("lw",   "I", "0000011", "010")
        self.opcodes.add("jalr", "I", "1100111", "000")

        # S-Type
        self.opcodes.add("sw",   "S", "0100011", "010")

        # B-Type
        self.opcodes.add("beq",  "B", "1100011", "000")
        self.opcodes.add("bne",  "B", "1100011", "001")

    # -------------------------------------------------
    # YARDIMCI FONKSİYONLAR
    # -------------------------------------------------
    def clean_line(self, line):
        line = self.remove_comment(line)
        return line.strip()

    def remove_comment(self, line):
        hash_pos = line.find("#")
        semicolon_pos = line.find(";")

        cut_pos = -1
        if hash_pos != -1 and semicolon_pos != -1:
            cut_pos = min(hash_pos, semicolon_pos)
        elif hash_pos != -1:
            cut_pos = hash_pos
        elif semicolon_pos != -1:
            cut_pos = semicolon_pos

        if cut_pos != -1:
            return line[:cut_pos]
        return line

    def split_label(self, line):
        if ":" in line:
            parts = line.split(":", 1)
            label = parts[0].strip()
            rest = parts[1].strip()
            self.validate_label_name(label)
            return label, rest
        return None, line

    def validate_label_name(self, label):
        if label == "":
            raise ValueError("Boş label tanımı")

        first = label[0]
        if not (first.isalpha() or first == "_"):
            raise ValueError(f"Geçersiz label adı: {label}")

        for ch in label:
            if not (ch.isalnum() or ch == "_"):
                raise ValueError(f"Geçersiz label adı: {label}")

    def is_directive(self, text):
        return text.startswith(".")

    def split_instruction(self, text):
        text = text.strip()
        if text == "":
            return "", ""

        first_space = text.find(" ")
        if first_space == -1:
            return text, ""

        mnemonic = text[:first_space].strip()
        operands_text = text[first_space + 1:].strip()
        return mnemonic, operands_text

    def split_operands(self, operand_text):
        result = []
        current = ""
        i = 0

        while i < len(operand_text):
            ch = operand_text[i]

            if ch == ",":
                result.append(current.strip())
                current = ""
            else:
                current += ch

            i += 1

        if current.strip() != "":
            result.append(current.strip())

        return result

    def parse_register(self, token):
        token = token.strip()

        if not token.startswith("x"):
            raise ValueError(f"Geçersiz register: {token}")

        num_str = token[1:]
        if not num_str.isdigit():
            raise ValueError(f"Geçersiz register: {token}")

        reg_num = int(num_str)
        if reg_num < 0 or reg_num > 31:
            raise ValueError(f"Register aralık dışı: {token}")

        return format(reg_num, "05b")

    def parse_immediate(self, token):
        token = token.strip()
        try:
            return int(token, 0)
        except Exception as exc:
            raise ValueError(f"Geçersiz immediate değer: {token}") from exc

    def check_signed_range(self, value, bits, field_name):
        low = -(1 << (bits - 1))
        high = (1 << (bits - 1)) - 1
        if value < low or value > high:
            raise ValueError(f"{field_name} değeri {bits} bit signed aralığında değil: {value}")

    def to_twos_complement(self, value, bits):
        mask = (1 << bits) - 1
        return value & mask

    def parse_memory_operand(self, text):
        text = text.strip()
        left_paren = text.find("(")
        right_paren = text.find(")")

        if left_paren == -1 or right_paren == -1 or right_paren < left_paren:
            raise ValueError(f"Geçersiz bellek operandı: {text}")

        imm_text = text[:left_paren].strip()
        reg_text = text[left_paren + 1:right_paren].strip()

        if imm_text == "":
            imm_value = 0
        else:
            imm_value = self.parse_immediate(imm_text)

        rs1 = self.parse_register(reg_text)
        return imm_value, rs1

    def instruction_size(self, text):
        if text.startswith(".word"):
            return 4
        if text.startswith(".byte"):
            return 1
        if text.startswith(".data"):
            return 0
        if text.startswith(".text"):
            return 0
        if text.startswith(".org"):
            return 0
        if text.startswith(".end"):
            return 0
        return 4

    def bin_to_hex(self, binary_text):
        if binary_text == "":
            return ""
        value = int(binary_text, 2)
        width = (len(binary_text) + 3) // 4
        return f"0x{value:0{width}X}"

    def format_binary_grouped(self, binary_text):
        grouped = ""
        i = 0
        while i < len(binary_text):
            grouped += binary_text[i:i + 4]
            if i + 4 < len(binary_text):
                grouped += " "
            i += 4
        return grouped

    def resolve_branch_offset(self, label_or_imm, pc):
        symbol = self.symbols.find(label_or_imm)
        if symbol is not None:
            return symbol.address - pc

        return self.parse_immediate(label_or_imm)

    # -------------------------------------------------
    # PASS 1
    # -------------------------------------------------
    def pass1(self, input_file):
        pc = self.start_address
        self.current_segment = ".text"

        with open(input_file, "r", encoding="utf-8") as file_handle:
            line_no = 0
            for raw_line in file_handle:
                line_no += 1
                line = self.clean_line(raw_line)

                if line == "":
                    continue

                label, rest = self.split_label(line)

                if label is not None:
                    self.symbols.add(label, pc)

                if rest == "":
                    continue

                if self.is_directive(rest):
                    if rest.startswith(".org"):
                        parts = rest.split(maxsplit=1)
                        if len(parts) != 2:
                            raise ValueError(f"Satır {line_no}: .org parametresi eksik")
                        pc = self.parse_immediate(parts[1])
                        continue

                    if rest.startswith(".data"):
                        self.current_segment = ".data"
                        continue

                    if rest.startswith(".text"):
                        self.current_segment = ".text"
                        continue

                    if rest.startswith(".end"):
                        break

                    pc += self.instruction_size(rest)
                    continue

                mnemonic, operands_text = self.split_instruction(rest)
                op_info = self.opcodes.find(mnemonic)
                if op_info is None:
                    raise ValueError(f"Satır {line_no}: Geçersiz komut: {mnemonic}")

                pc += 4

    # -------------------------------------------------
    # ENCODE
    # -------------------------------------------------
    def encode_r_type(self, op_info, operands):
        if len(operands) != 3:
            raise ValueError("R-Type komut için 3 operand gerekli")

        rd = self.parse_register(operands[0])
        rs1 = self.parse_register(operands[1])
        rs2 = self.parse_register(operands[2])

        return (
            op_info.funct7 +
            rs2 +
            rs1 +
            op_info.funct3 +
            rd +
            op_info.opcode
        )

    def encode_i_type(self, mnemonic, op_info, operands):
        if mnemonic == "lw":
            if len(operands) != 2:
                raise ValueError("lw için 2 operand gerekli")

            rd = self.parse_register(operands[0])
            imm_value, rs1 = self.parse_memory_operand(operands[1])

            self.check_signed_range(imm_value, 12, "lw immediate")
            imm = format(self.to_twos_complement(imm_value, 12), "012b")

            return imm + rs1 + op_info.funct3 + rd + op_info.opcode

        if mnemonic == "addi":
            if len(operands) != 3:
                raise ValueError("addi için 3 operand gerekli")

            rd = self.parse_register(operands[0])
            rs1 = self.parse_register(operands[1])
            imm_value = self.parse_immediate(operands[2])

            self.check_signed_range(imm_value, 12, "addi immediate")
            imm = format(self.to_twos_complement(imm_value, 12), "012b")

            return imm + rs1 + op_info.funct3 + rd + op_info.opcode

        if mnemonic == "jalr":
            if len(operands) != 3:
                raise ValueError("jalr için 3 operand gerekli")

            rd = self.parse_register(operands[0])
            rs1 = self.parse_register(operands[1])
            imm_value = self.parse_immediate(operands[2])

            self.check_signed_range(imm_value, 12, "jalr immediate")
            imm = format(self.to_twos_complement(imm_value, 12), "012b")

            return imm + rs1 + op_info.funct3 + rd + op_info.opcode

        raise ValueError(f"Desteklenmeyen I-Type komut: {mnemonic}")

    def encode_s_type(self, op_info, operands):
        if len(operands) != 2:
            raise ValueError("S-Type komut için 2 operand gerekli")

        rs2 = self.parse_register(operands[0])
        imm_value, rs1 = self.parse_memory_operand(operands[1])

        self.check_signed_range(imm_value, 12, "sw immediate")
        imm_bin = format(self.to_twos_complement(imm_value, 12), "012b")

        imm_high = imm_bin[:7]
        imm_low = imm_bin[7:]

        return (
            imm_high +
            rs2 +
            rs1 +
            op_info.funct3 +
            imm_low +
            op_info.opcode
        )

    def encode_b_type(self, op_info, operands, pc):
        if len(operands) != 3:
            raise ValueError("B-Type komut için 3 operand gerekli")

        rs1 = self.parse_register(operands[0])
        rs2 = self.parse_register(operands[1])

        label_or_imm = operands[2].strip()
        offset = self.resolve_branch_offset(label_or_imm, pc)

        self.check_signed_range(offset, 13, "branch offset")
        imm13 = self.to_twos_complement(offset, 13)

        bit12 = (imm13 >> 12) & 1
        bit11 = (imm13 >> 11) & 1
        bits10_5 = (imm13 >> 5) & 0b111111
        bits4_1 = (imm13 >> 1) & 0b1111

        return (
            format(bit12, "01b") +
            format(bits10_5, "06b") +
            rs2 +
            rs1 +
            op_info.funct3 +
            format(bits4_1, "04b") +
            format(bit11, "01b") +
            op_info.opcode
        )

    def encode_instruction(self, text, pc):
        mnemonic, operands_text = self.split_instruction(text)
        op_info = self.opcodes.find(mnemonic)

        if op_info is None:
            raise ValueError(f"Geçersiz komut: {mnemonic}")

        operands = []
        if operands_text != "":
            operands = self.split_operands(operands_text)

        if op_info.fmt == "R":
            return self.encode_r_type(op_info, operands)

        if op_info.fmt == "I":
            return self.encode_i_type(mnemonic, op_info, operands)

        if op_info.fmt == "S":
            return self.encode_s_type(op_info, operands)

        if op_info.fmt == "B":
            return self.encode_b_type(op_info, operands, pc)

        raise ValueError(f"Bilinmeyen komut formatı: {op_info.fmt}")

    # -------------------------------------------------
    # DIRECTIVE PASS 2
    # -------------------------------------------------
    def process_directive_pass2(self, text, output_handle, pc):
        if text.startswith(".data") or text.startswith(".text"):
            return pc, False

        if text.startswith(".org"):
            parts = text.split(maxsplit=1)
            if len(parts) != 2:
                raise ValueError(".org parametresi eksik")
            pc = self.parse_immediate(parts[1])
            return pc, False

        if text.startswith(".word"):
            parts = text.split(maxsplit=1)
            if len(parts) != 2:
                raise ValueError(".word parametresi eksik")

            value = self.parse_immediate(parts[1])
            self.check_signed_range(value, 32, ".word")
            binary = format(self.to_twos_complement(value, 32), "032b")
            hex_value = self.bin_to_hex(binary)

            output_handle.write(
                f"{pc:08X} | DIRECTIVE | .word | {self.format_binary_grouped(binary)} | {hex_value}\n"
            )
            return pc + 4, False

        if text.startswith(".byte"):
            parts = text.split(maxsplit=1)
            if len(parts) != 2:
                raise ValueError(".byte parametresi eksik")

            value = self.parse_immediate(parts[1])
            self.check_signed_range(value, 8, ".byte")
            binary = format(self.to_twos_complement(value, 8), "08b")
            hex_value = self.bin_to_hex(binary)

            output_handle.write(
                f"{pc:08X} | DIRECTIVE | .byte | {self.format_binary_grouped(binary)} | {hex_value}\n"
            )
            return pc + 1, False

        if text.startswith(".end"):
            return pc, True

        raise ValueError(f"Desteklenmeyen direktif: {text}")

    # -------------------------------------------------
    # PASS 2
    # -------------------------------------------------
    def pass2(self, input_file, output_file):
        pc = self.start_address

        with open(input_file, "r", encoding="utf-8") as fin, open(output_file, "w", encoding="utf-8") as fout:
            fout.write("ADDRESS  | TYPE      | SOURCE                  | BINARY                                | HEX\n")
            fout.write("-" * 105 + "\n")

            line_no = 0
            for raw_line in fin:
                line_no += 1
                line = self.clean_line(raw_line)

                if line == "":
                    continue

                label, rest = self.split_label(line)

                if label is not None:
                    text = rest.strip()
                else:
                    text = line

                if text == "":
                    continue

                try:
                    if self.is_directive(text):
                        pc, should_stop = self.process_directive_pass2(text, fout, pc)
                        if should_stop:
                            break
                        continue

                    machine_code = self.encode_instruction(text, pc)
                    hex_code = self.bin_to_hex(machine_code)

                    fout.write(
                        f"{pc:08X} | INSTRUCTION | {text:22} | {self.format_binary_grouped(machine_code):35} | {hex_code}\n"
                    )

                    pc += 4

                except Exception as exc:
                    raise ValueError(f"Satır {line_no}: {exc}") from exc

    # -------------------------------------------------
    # ANA İŞLEM
    # -------------------------------------------------
    def assemble(self, input_file, output_file="output.txt"):
        if not os.path.exists(input_file):
            raise FileNotFoundError(f"Girdi dosyası bulunamadı: {input_file}")

        self.pass1(input_file)
        self.pass2(input_file, output_file)

        print("\nAssembly işlemi tamamlandı.")
        print(f"Girdi dosyası : {input_file}")
        print(f"Çıktı dosyası : {output_file}")
        self.symbols.display()

    # -------------------------------------------------
    # RAPOR / SUNUM İÇİN EK GÖSTERİM
    # -------------------------------------------------
    def show_tables(self):
        self.opcodes.display()
        self.symbols.display()

In [16]:
assembler = Assembler()
assembler.assemble("input.asm", "output.txt")


Assembly işlemi tamamlandı.
Girdi dosyası : input.asm
Çıktı dosyası : output.txt

Symbol Table
----------------------------------------
Label                Address        
----------------------------------------
start                0x00000000
val1                 0x0000001C
val2                 0x00000020
end                  0x00000021
----------------------------------------


In [18]:
import os


# =========================================================
# LINKED LIST TABANLI OPCODE TABLE
# =========================================================
class OpcodeNode:
    def __init__(self, mnemonic, fmt, opcode, funct3="", funct7=""):
        self.mnemonic = mnemonic
        self.fmt = fmt
        self.opcode = opcode
        self.funct3 = funct3
        self.funct7 = funct7
        self.next = None


class OpcodeTable:
    def __init__(self):
        self.head = None

    def add(self, mnemonic, fmt, opcode, funct3="", funct7=""):
        new_node = OpcodeNode(mnemonic, fmt, opcode, funct3, funct7)

        if self.head is None:
            self.head = new_node
            return

        current = self.head
        while current.next is not None:
            current = current.next
        current.next = new_node

    def find(self, mnemonic):
        current = self.head
        while current is not None:
            if current.mnemonic == mnemonic:
                return current
            current = current.next
        return None

    def display(self):
        print("\nOpcode Table")
        print("-" * 75)
        print(f"{'Mnemonic':10} {'Format':8} {'Opcode':10} {'funct3':8} {'funct7':10}")
        print("-" * 75)

        current = self.head
        while current is not None:
            print(
                f"{current.mnemonic:10} "
                f"{current.fmt:8} "
                f"{current.opcode:10} "
                f"{current.funct3:8} "
                f"{current.funct7:10}"
            )
            current = current.next

        print("-" * 75)


# =========================================================
# LINKED LIST TABANLI SYMBOL TABLE
# =========================================================
class SymbolNode:
    def __init__(self, label, address):
        self.label = label
        self.address = address
        self.next = None


class SymbolTable:
    def __init__(self):
        self.head = None

    def add(self, label, address):
        if self.find(label) is not None:
            raise ValueError(f"Yinelenen label: {label}")

        new_node = SymbolNode(label, address)

        if self.head is None:
            self.head = new_node
            return

        current = self.head
        while current.next is not None:
            current = current.next
        current.next = new_node

    def find(self, label):
        current = self.head
        while current is not None:
            if current.label == label:
                return current
            current = current.next
        return None

    def display(self):
        print("\nSymbol Table")
        print("-" * 45)
        print(f"{'Label':20} {'Address':15}")
        print("-" * 45)

        current = self.head
        while current is not None:
            print(f"{current.label:20} 0x{current.address:08X}")
            current = current.next

        print("-" * 45)


# =========================================================
# ASSEMBLER
# =========================================================
class Assembler:
    def __init__(self):
        self.opcodes = OpcodeTable()
        self.symbols = SymbolTable()
        self.start_address = 0
        self.current_segment = ".text"
        self._load_opcodes()

    # -----------------------------------------------------
    # OPCODE TABLE YÜKLEME
    # -----------------------------------------------------
    def _load_opcodes(self):
        # R-Type
        self.opcodes.add("add", "R", "0110011", "000", "0000000")
        self.opcodes.add("sub", "R", "0110011", "000", "0100000")
        self.opcodes.add("and", "R", "0110011", "111", "0000000")
        self.opcodes.add("or",  "R", "0110011", "110", "0000000")
        self.opcodes.add("xor", "R", "0110011", "100", "0000000")
        self.opcodes.add("sll", "R", "0110011", "001", "0000000")
        self.opcodes.add("srl", "R", "0110011", "101", "0000000")

        # I-Type
        self.opcodes.add("addi", "I", "0010011", "000")
        self.opcodes.add("lw",   "I", "0000011", "010")
        self.opcodes.add("jalr", "I", "1100111", "000")

        # S-Type
        self.opcodes.add("sw", "S", "0100011", "010")

        # B-Type
        self.opcodes.add("beq", "B", "1100011", "000")
        self.opcodes.add("bne", "B", "1100011", "001")

    # -----------------------------------------------------
    # YARDIMCI FONKSİYONLAR
    # -----------------------------------------------------
    def clean_line(self, line):
        line = self.remove_comment(line)
        return line.strip()

    def remove_comment(self, line):
        hash_pos = line.find("#")
        semicolon_pos = line.find(";")

        cut_pos = -1
        if hash_pos != -1 and semicolon_pos != -1:
            cut_pos = min(hash_pos, semicolon_pos)
        elif hash_pos != -1:
            cut_pos = hash_pos
        elif semicolon_pos != -1:
            cut_pos = semicolon_pos

        if cut_pos != -1:
            return line[:cut_pos]
        return line

    def split_label(self, line):
        if ":" in line:
            parts = line.split(":", 1)
            label = parts[0].strip()
            rest = parts[1].strip()
            self.validate_label_name(label)
            return label, rest
        return None, line

    def validate_label_name(self, label):
        if label == "":
            raise ValueError("Boş label tanımı")

        first = label[0]
        if not (first.isalpha() or first == "_"):
            raise ValueError(f"Geçersiz label adı: {label}")

        for ch in label:
            if not (ch.isalnum() or ch == "_"):
                raise ValueError(f"Geçersiz label adı: {label}")

    def is_directive(self, text):
        return text.startswith(".")

    def align_to_4(self, value):
        if value % 4 == 0:
            return value
        return value + (4 - (value % 4))

    def split_instruction(self, text):
        text = text.strip()
        if text == "":
            return "", ""

        first_space = text.find(" ")
        if first_space == -1:
            return text, ""

        mnemonic = text[:first_space].strip()
        operands_text = text[first_space + 1:].strip()
        return mnemonic, operands_text

    def split_operands(self, operand_text):
        result = []
        current = ""
        i = 0

        while i < len(operand_text):
            ch = operand_text[i]
            if ch == ",":
                result.append(current.strip())
                current = ""
            else:
                current += ch
            i += 1

        if current.strip() != "":
            result.append(current.strip())

        return result

    def parse_register(self, token):
        token = token.strip()

        if not token.startswith("x"):
            raise ValueError(f"Geçersiz register: {token}")

        num_str = token[1:]
        if not num_str.isdigit():
            raise ValueError(f"Geçersiz register: {token}")

        reg_num = int(num_str)
        if reg_num < 0 or reg_num > 31:
            raise ValueError(f"Register aralık dışı: {token}")

        return format(reg_num, "05b")

    def parse_immediate(self, token):
        token = token.strip()
        try:
            return int(token, 0)
        except Exception as exc:
            raise ValueError(f"Geçersiz immediate değer: {token}") from exc

    def parse_memory_operand(self, text):
        text = text.strip()
        left_paren = text.find("(")
        right_paren = text.find(")")

        if left_paren == -1 or right_paren == -1 or right_paren < left_paren:
            raise ValueError(f"Geçersiz bellek operandı: {text}")

        imm_text = text[:left_paren].strip()
        reg_text = text[left_paren + 1:right_paren].strip()

        if imm_text == "":
            imm_value = 0
        else:
            imm_value = self.parse_immediate(imm_text)

        rs1 = self.parse_register(reg_text)
        return imm_value, rs1

    def check_signed_range(self, value, bits, field_name):
        low = -(1 << (bits - 1))
        high = (1 << (bits - 1)) - 1
        if value < low or value > high:
            raise ValueError(f"{field_name} değeri {bits} bit signed aralığında değil: {value}")

    def to_twos_complement(self, value, bits):
        mask = (1 << bits) - 1
        return value & mask

    def bin_to_hex(self, binary_text):
        if binary_text == "":
            return ""
        value = int(binary_text, 2)
        width = (len(binary_text) + 3) // 4
        return f"0x{value:0{width}X}"

    def format_binary_grouped(self, binary_text):
        grouped = ""
        i = 0
        while i < len(binary_text):
            grouped += binary_text[i:i + 4]
            if i + 4 < len(binary_text):
                grouped += " "
            i += 4
        return grouped

    def instruction_size(self, text):
        if text.startswith(".word"):
            return 4
        if text.startswith(".byte"):
            return 1
        if text.startswith(".data"):
            return 0
        if text.startswith(".text"):
            return 0
        if text.startswith(".org"):
            return 0
        if text.startswith(".end"):
            return 0
        return 4

    # -----------------------------------------------------
    # PASS 1
    # -----------------------------------------------------
    def pass1(self, input_file):
        pc = self.start_address
        self.current_segment = ".text"

        with open(input_file, "r", encoding="utf-8") as file_handle:
            line_no = 0
            for raw_line in file_handle:
                line_no += 1
                line = self.clean_line(raw_line)

                if line == "":
                    continue

                label, rest = self.split_label(line)

                if label is not None:
                    self.symbols.add(label, pc)

                if rest == "":
                    continue

                if self.is_directive(rest):
                    if rest.startswith(".org"):
                        parts = rest.split(maxsplit=1)
                        if len(parts) != 2:
                            raise ValueError(f"Satır {line_no}: .org parametresi eksik")
                        pc = self.parse_immediate(parts[1])
                        continue

                    if rest.startswith(".data"):
                        self.current_segment = ".data"
                        continue

                    if rest.startswith(".text"):
                        self.current_segment = ".text"
                        pc = self.align_to_4(pc)
                        continue

                    if rest.startswith(".end"):
                        break

                    pc += self.instruction_size(rest)
                    continue

                mnemonic, _ = self.split_instruction(rest)
                op_info = self.opcodes.find(mnemonic)
                if op_info is None:
                    raise ValueError(f"Satır {line_no}: Geçersiz komut: {mnemonic}")

                pc += 4

    # -----------------------------------------------------
    # ENCODE FONKSİYONLARI
    # -----------------------------------------------------
    def encode_r_type(self, op_info, operands):
        if len(operands) != 3:
            raise ValueError("R-Type komut için 3 operand gerekli")

        rd = self.parse_register(operands[0])
        rs1 = self.parse_register(operands[1])
        rs2 = self.parse_register(operands[2])

        return (
            op_info.funct7 +
            rs2 +
            rs1 +
            op_info.funct3 +
            rd +
            op_info.opcode
        )

    def encode_i_type(self, mnemonic, op_info, operands):
        if mnemonic == "lw":
            if len(operands) != 2:
                raise ValueError("lw için 2 operand gerekli")

            rd = self.parse_register(operands[0])
            imm_value, rs1 = self.parse_memory_operand(operands[1])

            self.check_signed_range(imm_value, 12, "lw immediate")
            imm = format(self.to_twos_complement(imm_value, 12), "012b")

            return imm + rs1 + op_info.funct3 + rd + op_info.opcode

        if mnemonic == "addi":
            if len(operands) != 3:
                raise ValueError("addi için 3 operand gerekli")

            rd = self.parse_register(operands[0])
            rs1 = self.parse_register(operands[1])
            imm_value = self.parse_immediate(operands[2])

            self.check_signed_range(imm_value, 12, "addi immediate")
            imm = format(self.to_twos_complement(imm_value, 12), "012b")

            return imm + rs1 + op_info.funct3 + rd + op_info.opcode

        if mnemonic == "jalr":
            if len(operands) != 3:
                raise ValueError("jalr için 3 operand gerekli")

            rd = self.parse_register(operands[0])
            rs1 = self.parse_register(operands[1])
            imm_value = self.parse_immediate(operands[2])

            self.check_signed_range(imm_value, 12, "jalr immediate")
            imm = format(self.to_twos_complement(imm_value, 12), "012b")

            return imm + rs1 + op_info.funct3 + rd + op_info.opcode

        raise ValueError(f"Desteklenmeyen I-Type komut: {mnemonic}")

    def encode_s_type(self, op_info, operands):
        if len(operands) != 2:
            raise ValueError("S-Type komut için 2 operand gerekli")

        rs2 = self.parse_register(operands[0])
        imm_value, rs1 = self.parse_memory_operand(operands[1])

        self.check_signed_range(imm_value, 12, "sw immediate")
        imm_bin = format(self.to_twos_complement(imm_value, 12), "012b")

        imm_high = imm_bin[:7]
        imm_low = imm_bin[7:]

        return (
            imm_high +
            rs2 +
            rs1 +
            op_info.funct3 +
            imm_low +
            op_info.opcode
        )

    def encode_b_type(self, op_info, operands, pc):
        if len(operands) != 3:
            raise ValueError("B-Type komut için 3 operand gerekli")

        rs1 = self.parse_register(operands[0])
        rs2 = self.parse_register(operands[1])

        target = operands[2].strip()

        symbol = self.symbols.find(target)
        if symbol is not None:
            offset = symbol.address - pc
        else:
            offset = self.parse_immediate(target)

        self.check_signed_range(offset, 13, "branch offset")
        imm13 = self.to_twos_complement(offset, 13)

        bit12 = (imm13 >> 12) & 1
        bit11 = (imm13 >> 11) & 1
        bits10_5 = (imm13 >> 5) & 0b111111
        bits4_1 = (imm13 >> 1) & 0b1111

        return (
            format(bit12, "01b") +
            format(bits10_5, "06b") +
            rs2 +
            rs1 +
            op_info.funct3 +
            format(bits4_1, "04b") +
            format(bit11, "01b") +
            op_info.opcode
        )

    def encode_instruction(self, text, pc):
        mnemonic, operands_text = self.split_instruction(text)
        op_info = self.opcodes.find(mnemonic)

        if op_info is None:
            raise ValueError(f"Geçersiz komut: {mnemonic}")

        operands = []
        if operands_text != "":
            operands = self.split_operands(operands_text)

        if op_info.fmt == "R":
            return self.encode_r_type(op_info, operands)

        if op_info.fmt == "I":
            return self.encode_i_type(mnemonic, op_info, operands)

        if op_info.fmt == "S":
            return self.encode_s_type(op_info, operands)

        if op_info.fmt == "B":
            return self.encode_b_type(op_info, operands, pc)

        raise ValueError(f"Bilinmeyen komut formatı: {op_info.fmt}")

    # -----------------------------------------------------
    # DIRECTIVE PASS 2
    # -----------------------------------------------------
    def process_directive_pass2(self, text, output_handle, pc):
        if text.startswith(".data"):
            return pc, False

        if text.startswith(".text"):
            pc = self.align_to_4(pc)
            return pc, False

        if text.startswith(".org"):
            parts = text.split(maxsplit=1)
            if len(parts) != 2:
                raise ValueError(".org parametresi eksik")
            pc = self.parse_immediate(parts[1])
            return pc, False

        if text.startswith(".word"):
            parts = text.split(maxsplit=1)
            if len(parts) != 2:
                raise ValueError(".word parametresi eksik")

            value = self.parse_immediate(parts[1])
            binary = format(self.to_twos_complement(value, 32), "032b")
            hex_value = self.bin_to_hex(binary)

            output_handle.write(
                f"{pc:08X} | DIRECTIVE   | .word {' ' + parts[1]:17} | {self.format_binary_grouped(binary):35} | {hex_value}\n"
            )
            return pc + 4, False

        if text.startswith(".byte"):
            parts = text.split(maxsplit=1)
            if len(parts) != 2:
                raise ValueError(".byte parametresi eksik")

            value = self.parse_immediate(parts[1])
            self.check_signed_range(value, 8, ".byte")
            binary = format(self.to_twos_complement(value, 8), "08b")
            hex_value = self.bin_to_hex(binary)

            output_handle.write(
                f"{pc:08X} | DIRECTIVE   | .byte {' ' + parts[1]:17} | {self.format_binary_grouped(binary):35} | {hex_value}\n"
            )
            return pc + 1, False

        if text.startswith(".end"):
            return pc, True

        raise ValueError(f"Desteklenmeyen direktif: {text}")

    # -----------------------------------------------------
    # PASS 2
    # -----------------------------------------------------
    def pass2(self, input_file, output_file):
        pc = self.start_address

        with open(input_file, "r", encoding="utf-8") as fin, open(output_file, "w", encoding="utf-8") as fout:
            fout.write("ADDRESS  | TYPE        | SOURCE                 | BINARY                                | HEX\n")
            fout.write("-" * 108 + "\n")

            line_no = 0
            for raw_line in fin:
                line_no += 1
                line = self.clean_line(raw_line)

                if line == "":
                    continue

                label, rest = self.split_label(line)
                text = rest.strip() if label is not None else line

                if text == "":
                    continue

                try:
                    if self.is_directive(text):
                        pc, should_stop = self.process_directive_pass2(text, fout, pc)
                        if should_stop:
                            break
                        continue

                    machine_code = self.encode_instruction(text, pc)
                    hex_code = self.bin_to_hex(machine_code)

                    fout.write(
                        f"{pc:08X} | INSTRUCTION | {text:22} | {self.format_binary_grouped(machine_code):35} | {hex_code}\n"
                    )

                    pc += 4

                except Exception as exc:
                    raise ValueError(f"Satır {line_no}: {exc}") from exc

    # -----------------------------------------------------
    # ANA ÇALIŞTIRMA
    # -----------------------------------------------------
    def assemble(self, input_file, output_file="output.txt"):
        if not os.path.exists(input_file):
            raise FileNotFoundError(f"Girdi dosyası bulunamadı: {input_file}")

        self.pass1(input_file)
        self.pass2(input_file, output_file)

        print("\nAssembly işlemi tamamlandı.")
        print(f"Girdi dosyası : {input_file}")
        print(f"Çıktı dosyası : {output_file}")
        self.symbols.display()

    # -----------------------------------------------------
    # EK GÖSTERİM
    # -----------------------------------------------------
    def show_tables(self):
        self.opcodes.display()
        self.symbols.display()

In [20]:
asm_code = """
.org 0x0000
.text

start:  add x1, x2, x3
        sub x4, x1, x5
        addi x6, x0, 10
        lw x7, 0(x1)
        sw x7, 4(x1)
        beq x1, x2, end
        bne x1, x3, start

.data
val1:   .word 25
val2:   .byte 7

.text
end:    or x8, x1, x2
        .end
"""

with open("input.asm", "w", encoding="utf-8") as f:
    f.write(asm_code)

print("input.asm oluşturuldu")

input.asm oluşturuldu


In [22]:
assembler = Assembler()
assembler.assemble("input.asm", "output.txt")


Assembly işlemi tamamlandı.
Girdi dosyası : input.asm
Çıktı dosyası : output.txt

Symbol Table
---------------------------------------------
Label                Address        
---------------------------------------------
start                0x00000000
val1                 0x0000001C
val2                 0x00000020
end                  0x00000024
---------------------------------------------


In [24]:
assembler.show_tables()


Opcode Table
---------------------------------------------------------------------------
Mnemonic   Format   Opcode     funct3   funct7    
---------------------------------------------------------------------------
add        R        0110011    000      0000000   
sub        R        0110011    000      0100000   
and        R        0110011    111      0000000   
or         R        0110011    110      0000000   
xor        R        0110011    100      0000000   
sll        R        0110011    001      0000000   
srl        R        0110011    101      0000000   
addi       I        0010011    000                
lw         I        0000011    010                
jalr       I        1100111    000                
sw         S        0100011    010                
beq        B        1100011    000                
bne        B        1100011    001                
---------------------------------------------------------------------------

Symbol Table
------------------------------

In [26]:
with open("output.txt", "r", encoding="utf-8") as f:
    print(f.read())

ADDRESS  | TYPE        | SOURCE                 | BINARY                                | HEX
------------------------------------------------------------------------------------------------------------
00000000 | INSTRUCTION | add x1, x2, x3         | 0000 0000 0011 0001 0000 0000 1011 0011 | 0x003100B3
00000004 | INSTRUCTION | sub x4, x1, x5         | 0100 0000 0101 0000 1000 0010 0011 0011 | 0x40508233
00000008 | INSTRUCTION | addi x6, x0, 10        | 0000 0000 1010 0000 0000 0011 0001 0011 | 0x00A00313
0000000C | INSTRUCTION | lw x7, 0(x1)           | 0000 0000 0000 0000 1010 0011 1000 0011 | 0x0000A383
00000010 | INSTRUCTION | sw x7, 4(x1)           | 0000 0000 0111 0000 1010 0010 0010 0011 | 0x0070A223
00000014 | INSTRUCTION | beq x1, x2, end        | 0000 0000 0010 0000 1000 1000 0110 0011 | 0x00208863
00000018 | INSTRUCTION | bne x1, x3, start      | 1111 1110 0011 0000 1001 0100 1110 0011 | 0xFE3094E3
0000001C | DIRECTIVE   | .word  25               | 0000 0000 0000 0000 0000 